# 04 - Coherence proof of concept (one site, one year)

**Question:** does Sentinel-1 *coherence* separate built-up, bare ground and vegetation better than the radar brightness (VV, VH) our model uses, and does it add anything on top of brightness?

**Coherence** compares two radar passes 12 days apart, pixel by pixel, using the phase of the wave (which our brightness images do not contain). 1 = the pixel looks exactly the same in both passes (buildings, rock), 0 = it changed completely (vegetation moving, water). It is still Sentinel-1 data, so it fits the brief's "Sentinel-1 only".

**How:** NASA's Alaska Satellite Facility computes coherence for us with its HyP3 service, one job per pair of passes and radar "burst" (a strip of one satellite pass). We take every pair 12 days apart over one training site for 2024, average them per month, align them to the exact 10 m grid of our radar files, and compare per class using the Dynamic World monthly labels.

**Before running:** a NASA Earthdata login stored on the server (`server_setup.sh` restores it), and `COHERENCE_RAW` pointing at the fast NVMe disk if you have one (the downloads are large; the aligned monthly maps are small and go to `data/coherence/`).

## 1. Settings

In [ ]:
import collections
import csv
import glob
import os
import zipfile
from datetime import datetime, timedelta

import asf_search as asf
import hyp3_sdk
import matplotlib.pyplot as plt
import numpy as np
import rasterio
from matplotlib.colors import ListedColormap
from rasterio.warp import Resampling, reproject
from rasterio.warp import transform as warp_points

from lulc import CLASSES, COLORS, IGNORE, REMAP

SITE = "pakokku"                  # training site: town, bare ground, crops and a river side by side
YEAR = 2024
LOOKS = "5x1"                     # finest HyP3 option, about 20 m pixels (10x2 = 40 m, 20x4 = 80 m)
JOB_NAME = f"coh_{SITE}_{YEAR}"   # every job of this site-year carries this name, so a rerun finds them instead of paying twice
RAW_DIR = os.environ.get("COHERENCE_RAW", "coherence_raw")   # the big downloads: point this at the fast NVMe disk
OUT_DIR = "data/coherence"        # the small aligned monthly maps
SUBMIT = False                    # set True only after checking the pairs and the cost below
SIZE_PX, SCALE = 1000, 10

sites = {r["name"]: r for r in csv.DictReader(open("data/sites.csv"))}
LAT, LON = float(sites[SITE]["lat"]), float(sites[SITE]["lon"])
with np.load(f"data/{SITE}_{YEAR}-01.npz") as z:
    PASS = str(z["pass_dir"])     # the orbit direction our radar files of this site use
print(SITE, LAT, LON, PASS)

## 2. Account check

`application_status` should say the account may submit jobs, and there must be enough credits for the jobs in section 4. The cost table shows what one burst job costs.

In [ ]:
hyp3 = hyp3_sdk.HyP3()            # logs in with ~/.netrc
info = hyp3.my_info()
print({k: info.get(k) for k in ("application_status", "remaining_credits")})
print("credits left:", hyp3.check_credits())
print("cost per job:", {k: v for k, v in hyp3.costs().items() if "BURST" in k})

## 3. Which pairs of passes exist

The public ASF search lists every radar burst over the site's square in the same orbit direction as our radar files. Consecutive passes of the same burst, exactly 12 days apart, form a pair (a missing pass would make a 24-day pair, which decorrelates more, so it is skipped). The picture shows every pass (dots) and every pair (lines); a month's coherence is the average of its pairs.

In [ ]:
def site_grid(lat, lon):
    """The same 10 km UTM grid as the radar export (make_grid in 01_data_export.ipynb), without Earth Engine."""
    zone = int((lon + 180) // 6) + 1
    epsg = f"EPSG:{(32600 if lat >= 0 else 32700) + zone}"
    (x,), (y,) = warp_points("EPSG:4326", epsg, [lon], [lat])
    half = SIZE_PX * SCALE / 2
    return epsg, rasterio.Affine(SCALE, 0, x - half, 0, -SCALE, y + half)


CRS, AFFINE = site_grid(LAT, LON)
d = 0.05                                                         # about 5.5 km: the 10 km square plus a margin
wkt = f"POLYGON(({LON-d} {LAT-d},{LON+d} {LAT-d},{LON+d} {LAT+d},{LON-d} {LAT+d},{LON-d} {LAT-d}))"
found = asf.search(dataset=asf.DATASET.SLC_BURST, intersectsWith=wkt, start=f"{YEAR}-01-01", end=f"{YEAR + 1}-01-13",
                   polarization=asf.POLARIZATION.VV, flightDirection=PASS)
by_burst = collections.defaultdict(set)
for r in found:
    p = r.properties
    by_burst[p["burst"]["fullBurstID"]].add((p["startTime"][:10], p["sceneName"]))
pairs = []                        # (burst, reference date, reference scene, secondary scene)
for burst, acq in sorted(by_burst.items()):
    acq = sorted(acq)
    for (d1, s1), (d2, s2) in zip(acq, acq[1:]):
        if (datetime.fromisoformat(d2) - datetime.fromisoformat(d1)).days == 12 and d1.startswith(str(YEAR)):
            pairs.append((burst, d1, s1, s2))
print(f"{len(by_burst)} bursts, {sum(map(len, by_burst.values()))} passes, {len(pairs)} pairs 12 days apart")
print("pairs per month:", dict(sorted(collections.Counter(d1[:7] for _, d1, _, _ in pairs).items())))

fig, ax = plt.subplots(figsize=(13, 1.2 + 0.6 * len(by_burst)))
for i, (burst, acq) in enumerate(sorted(by_burst.items())):
    ax.scatter([datetime.fromisoformat(a) for a, _ in acq], [i] * len(acq), s=20, color="#52514e", zorder=3)
    for b, d1, _, _ in pairs:
        if b == burst:
            t = datetime.fromisoformat(d1)
            ax.plot([t, t + timedelta(days=12)], [i, i], color="#2a78d6", lw=3, alpha=0.6)
ax.set_yticks(range(len(by_burst)), sorted(by_burst))
ax.set_title(f"{SITE} {YEAR}, {PASS.lower()} passes: dots = passes, blue lines = the {len(pairs)} pairs to process", loc="left")
ax.grid(axis="x", alpha=0.3)
plt.tight_layout()
plt.show()

## 4. Submit the jobs (costs credits)

Jobs already submitted under `JOB_NAME` are found and not submitted again. With `SUBMIT = False` this cell only reports what it would do.

In [ ]:
existing = hyp3.find_jobs(name=JOB_NAME)
done = {tuple(j.job_parameters["granules"]) for j in existing}
todo = [(s1, s2) for _, _, s1, s2 in pairs if (s1, s2) not in done]
print(f"{len(existing)} jobs already submitted, {len(todo)} still to submit")
if todo and SUBMIT:
    batch = hyp3.submit_prepared_jobs([hyp3_sdk.HyP3.prepare_insar_isce_burst_job(s1, s2, name=JOB_NAME, looks=LOOKS) for s1, s2 in todo])
    print(f"submitted {len(batch)} jobs")
elif todo:
    print("SUBMIT is False: check the pairs and the cost above, then set SUBMIT = True and run this cell again")

## 5. Wait for the jobs and download the results

`watch` waits until every job has finished (often 10 to 60 minutes; the cell can be stopped and run again later). Products already on disk are not downloaded again. HyP3 keeps results for about two weeks.

In [ ]:
batch = hyp3.find_jobs(name=JOB_NAME)
if not batch.complete():
    batch = hyp3.watch(batch)
ok = [j for j in batch if j.succeeded()]
print(f"{len(ok)} of {len(batch)} jobs succeeded")
os.makedirs(RAW_DIR, exist_ok=True)
new = hyp3_sdk.Batch([j for j in ok if not os.path.exists(os.path.join(RAW_DIR, j.files[0]["filename"]))])
new.download_files(RAW_DIR)
print(f"downloaded {len(new)}, {len(ok) - len(new)} were already there")
ref_date = {s1: d1 for _, d1, s1, _ in pairs}
products = [(os.path.join(RAW_DIR, j.files[0]["filename"]), ref_date[j.job_parameters["granules"][0]])
            for j in ok if j.job_parameters["granules"][0] in ref_date]

## 6. Align to our grid and average per month

Each product's coherence map (`*_corr.tif`, about 20 m) is resampled onto the site's 10 m grid, the same grid as the radar and label files, so pixels line up exactly. A month's map is the average of its pairs and bursts (a burst covers only part of the square, so the bursts fill each other in).

In [ ]:
def coherence_on_grid(zip_path):
    """A product's coherence map on the site's 10 m grid (NaN where the burst does not cover the square)."""
    with zipfile.ZipFile(zip_path) as z:
        inner = next(n for n in z.namelist() if n.endswith("_corr.tif"))
    out = np.full((SIZE_PX, SIZE_PX), np.nan, np.float32)
    with rasterio.open(f"zip://{os.path.abspath(zip_path)}!/{inner}") as src:
        reproject(rasterio.band(src, 1), out, src_nodata=src.nodata if src.nodata is not None else 0,
                  dst_transform=AFFINE, dst_crs=CRS, dst_nodata=np.nan, resampling=Resampling.bilinear)
    return out


os.makedirs(OUT_DIR, exist_ok=True)
per_month = collections.defaultdict(list)
for zip_path, date in products:
    per_month[date[:7]].append(coherence_on_grid(zip_path))
for month, maps in sorted(per_month.items()):
    coh = np.nanmean(np.stack(maps), axis=0)
    np.savez_compressed(f"{OUT_DIR}/{SITE}_{month}.npz", coh=coh.astype(np.float16), n_pairs=len(maps))
    print(f"{month}: {len(maps)} maps, square covered {np.isfinite(coh).mean():.0%}, mean coherence {np.nanmean(coh):.2f}")

## 7. One month side by side

Brightness (VV, VH), coherence and the Dynamic World label of the same month. Buildings should be bright in coherence (stable between passes), fields and vegetation dark.

In [ ]:
MONTH = f"{YEAR}-02"              # a dry-season month (Pakokku's rains are about June to October)
with np.load(f"data/{SITE}_{MONTH}.npz") as r:
    radar, label = r["radar"], REMAP[r["label"]]
coh = np.load(f"{OUT_DIR}/{SITE}_{MONTH}.npz")["coh"].astype(np.float32)
fig, ax = plt.subplots(1, 4, figsize=(21, 5.6))
ax[0].imshow(radar[0], cmap="gray", vmin=-25, vmax=5); ax[0].set_title("VV brightness (dB)")
ax[1].imshow(radar[1], cmap="gray", vmin=-35, vmax=-5); ax[1].set_title("VH brightness (dB)")
im = ax[2].imshow(coh, cmap="magma", vmin=0, vmax=1); ax[2].set_title("coherence (12-day pairs)")
plt.colorbar(im, ax=ax[2], fraction=0.046)
ax[3].imshow(np.ma.masked_equal(label, IGNORE), cmap=ListedColormap(COLORS), vmin=-0.5, vmax=len(CLASSES) - 0.5, interpolation="nearest")
ax[3].set_title("Dynamic World label (white = none)")
for a in ax:
    a.set_xticks([]); a.set_yticks([])
fig.legend([plt.Rectangle((0, 0), 1, 1, color=c) for c in COLORS], CLASSES, loc="lower center", ncol=len(CLASSES))
fig.suptitle(f"{SITE}, {MONTH}", x=0.01, ha="left")
plt.tight_layout(rect=(0, 0.06, 1, 1))
plt.show()

## 8. The answer: does coherence separate the classes?

All months together, pixels with a real monthly label only. **Left:** each feature's spread per class. **Right:** for pairs of classes we confuse, how well each feature *alone* tells them apart (0.5 = not at all, 1 = perfectly). If coherence's bar is clearly taller for built vs bare or built vs low vegetation, it carries information brightness lacks.

**Added value:** the model never uses one feature alone, it combines VV and VH. So the last line trains a very simple classifier twice, on VV+VH and on VV+VH+coherence, and compares. Train and score pixels come from the same site, so both numbers are optimistic; only their difference matters.

In [ ]:
feats = {"VV (dB)": [], "VH (dB)": [], "coherence": []}
cls = []
for f in sorted(glob.glob(f"{OUT_DIR}/{SITE}_{YEAR}-*.npz")):
    month = os.path.basename(f)[len(SITE) + 1:-4]
    with np.load(f"data/{SITE}_{month}.npz") as r:
        radar, label = r["radar"], REMAP[r["label"]]
    coh = np.load(f)["coh"].astype(np.float32)
    keep = (label != IGNORE) & np.isfinite(coh) & (radar > -900).all(0)
    for name, values in zip(feats, (radar[0], radar[1], coh)):
        feats[name].append(values[keep])
    cls.append(label[keep])
feats = {k: np.concatenate(v) for k, v in feats.items()}
cls = np.concatenate(cls)
rng = np.random.default_rng(0)
sample = {c: rng.permutation(np.flatnonzero(cls == c))[:20000] for c in range(len(CLASSES)) if (cls == c).sum() >= 500}
present = [CLASSES[c] for c in sample]
print("pixels per class (all months):", {CLASSES[c]: int((cls == c).sum()) for c in sample})


def separation(a, b):
    """How well one feature alone tells two classes apart: 0.5 = not at all, 1 = perfectly (rank-based AUC)."""
    ranks = np.argsort(np.argsort(np.concatenate([a, b])))
    auc = (ranks[:len(a)].sum() - len(a) * (len(a) - 1) / 2) / (len(a) * len(b))
    return max(auc, 1 - auc)


def simple_accuracy(names):
    """Balanced accuracy of a per-class Gaussian classifier on these features: fit on half the sampled pixels, score on the rest."""
    fit, test = {}, {}
    for c, idx in sample.items():
        x = np.stack([feats[n][idx] for n in names], 1)
        half = len(x) // 2
        fit[c] = (x[:half].mean(0), x[:half].std(0) + 1e-6)
        test[c] = x[half:]
    recall = []
    for c, x in test.items():
        ll = np.stack([-(((x - m) / s) ** 2).sum(1) / 2 - np.log(s).sum() for m, s in fit.values()], 1)
        recall.append((np.array(list(fit))[ll.argmax(1)] == c).mean())
    return float(np.mean(recall))


C_ = {n: CLASSES.index(n) for n in present}
pairs_to_test = [(a, b) for a, b in [("built", "bare"), ("built", "low_vegetation"), ("built", "trees"), ("bare", "low_vegetation"),
                                     ("crops", "low_vegetation"), ("crops", "bare"), ("water", "bare")] if a in C_ and b in C_]
fig, ax = plt.subplots(1, 2, figsize=(18, 5), gridspec_kw={"width_ratios": [1.2, 1]})
pos = np.arange(len(present))
for i, (name, color) in enumerate(zip(feats, ("#86b6ef", "#2a78d6", "#eb6834"))):
    vals = [(feats[name][sample[C_[n]]] - feats[name].mean()) / feats[name].std() for n in present]   # standardised to compare on one axis
    ax[0].boxplot(vals, positions=pos + (i - 1) * 0.27, widths=0.22, showfliers=False, patch_artist=True,
                  boxprops=dict(facecolor=color, edgecolor="#52514e"), medianprops=dict(color="#0b0b0b"))
    ax[0].plot([], [], color=color, lw=8, label=name)
ax[0].set_xticks(pos, present, rotation=20, ha="right"); ax[0].set_ylabel("standardised value")
ax[0].legend(frameon=False); ax[0].set_title("each feature per class (boxes: middle 50% of pixels)", loc="left")
x = np.arange(len(pairs_to_test))
for i, (name, color) in enumerate(zip(feats, ("#86b6ef", "#2a78d6", "#eb6834"))):
    sep = [separation(feats[name][sample[C_[a]]], feats[name][sample[C_[b]]]) for a, b in pairs_to_test]
    ax[1].bar(x + (i - 1) * 0.27, sep, 0.25, color=color, edgecolor="white", label=name)
ax[1].set_xticks(x, [f"{a} vs {b}" for a, b in pairs_to_test], rotation=25, ha="right")
ax[1].set_ylim(0.5, 1); ax[1].set_ylabel("separation (0.5 = none, 1 = perfect)")
ax[1].legend(frameon=False); ax[1].set_title("how well one feature alone separates two classes", loc="left")
plt.tight_layout()
plt.show()

base, plus = simple_accuracy(["VV (dB)", "VH (dB)"]), simple_accuracy(["VV (dB)", "VH (dB)", "coherence"])
print(f"simple classifier, balanced accuracy over {len(sample)} classes: VV+VH {base:.3f}  ->  VV+VH+coherence {plus:.3f}  ({plus - base:+.3f})")